<u/>**第二次模型拟合 / Second-time model regression**</u>

此笔记本将修改上一个模型当中的因子，然后进行第二次回归。这次回归将删除5个解释力弱的宏观因子以及政策事件dummies，并添加cpi，balance of trade，GDP作为新的宏观因子以及添加导致2015-2017年汇率大幅变化的政策事件dummies。在执行接下来的python代码之前，数据已经完成时间戳统一和数据清洗，并存放在```data/2_regression_monthly```。

This notebook will modify the factors in the previous model and then run a second regression. This regression will remove five macro factors with weak explanatory power and the policy event dummies, and add CPI, balance of trade, and GDP as new macro factors, as well as policy event dummies that caused large exchange rate fluctuations from 2015 to 2017. Before executing the following Python code, the data have already undergone timestamp unification and data cleaning and are stored in `data/2_regression_monthly`. 

**数据清洗的方式介绍 / Data-cleaning process intro.**

这里会处理三个宏观因子的数据： [source](data/source/) 当中的 [SLE_hcpi_m.csv](data/cleaned/SLE_hcpi_m.csv) ， [SLEGDP.csv](data/cleaned/SLEGDP.csv) ， [SLEBalanceofTrade.csv](data/cleaned/SLEBalanceofTrade.csv) 。

1. 截断数据的时间，2012 年 1 月至 2026 年 6 月。2012年之前的数据以及2026年6月之后的数据（如果有）删除。
2. 查找空缺的数据，如果要是数据出现了历史数据的空缺（最后一个发布数据与第一个发布的数据之间的空缺），则使用前一个数值（注意这里应该是数值，不是变化率）进行线性插值（也就是，前一个数据和后一个数据之间的插值除以时间，得到的每一个周期的变量）。如果数据最后部分空缺，保留最后的空缺（例如：最后只有到25年4月的数据，没有到26年的数据，则保留25年4月之后的空缺，不做任何填充）。
3. 对三个数据进行一阶差分：计算cpi的月度对数变化量，以及gdp和balance of trade的年度基期绝对值变化率：(本年值−上年值)/|上年值|。
4. 之后，对 [SLEGDP.csv](data/cleaned/SLEGDP.csv) 和 [SLEBalanceofTrade.csv](data/cleaned/SLEBalanceofTrade.csv) 两个年度数据进行月度化数据拆分，即年度数据拆成月度数据，然后每个月填充当年的变化值数据（例如，2026-01-01对应25年的数据，则25年1月-12月填充同一个数字，也就是24-25年变化值）。
5. 保留空缺数据的时间戳需要对齐 [USDSLE_monthly.csv](data/2_regression_monthly/USDSLE_monthly.csv) 当中的时间戳。
6. 所有数据存放在```data/2_regression_monthly```文件夹当中。


This section processes data for three macroeconomic factors: [SLE_hcpi_m.csv](data/cleaned/SLE_hcpi_m.csv), [SLEGDP.csv](data/cleaned/SLEGDP.csv), and [SLEBalanceofTrade.csv](data/cleaned/SLEBalanceofTrade.csv) in [source](data/source/).

1. Truncate the data to the period from January 2012 to June 2026. Delete data before 2012 and data after June 2026 (if any).
2. Search for missing data. If there is a historical gap in the data (a gap between the last published observation and the first published observation), fill it by linear interpolation using the adjacent values (note: interpolate the level, not the growth rate)—that is, interpolate between the previous and next observations over time to obtain a value for each period. If the missing data are at the end of the series, keep the trailing missing values (for example, if the last observation is only through April 2025 and there is no data through 2026, keep the missing values after April 2025 and do not fill them).
3. Take the first difference of the three series: calculate the monthly log change in CPI, and the annual base-period absolute-value change rate for GDP and balance of trade: \((current year value - previous year value)/|previous year value|\).
4. Next, perform monthly disaggregation of the two annual series, [SLEGDP.csv](data/cleaned/SLEGDP.csv) and [SLEBalanceofTrade.csv](data/cleaned/SLEBalanceofTrade.csv); that is, split the annual data into monthly data, and then fill each month with the change value for that year (for example, 2026-01-01 corresponds to 2025 data, so fill January–December 2025 with the same number, i.e., the 2024–25 change value).
5. For timestamps where missing data are retained, align them with the timestamps in [USDSLE_monthly.csv](data/2_regression_monthly/USDSLE_monthly.csv).
6. Store all data in the `data/2_regression_monthly` folder.

**政策与事件dummy整理/ Policy and Event Dummies**

| 变量 / Variable | 日期 / Date | 中文解释 | English explanation |
|---|---|---|---|
| `D_IMF_ECF_REVIEW_2015` | [2015-11-16](https://www.imf.org/en/news/articles/2015/09/14/01/49/pr15521) | IMF完成ECF第3、4次审查，批准4,614万美元拨款。 | The IMF completed the third and fourth ECF reviews and approved a $46.14 million disbursement. |
| `D_Brexit referendum_2016` | [2016-06-23](https://commonslibrary.parliament.uk/research-briefings/cbp-7960/) | 英国举行脱欧公投，脱欧阵营获胜。 | The UK held the Brexit referendum, with Leave winning. |
| `D_REDENOM_ANNOUNCE_2021` | [2021-08-11](https://bsl.gov.sl/Public%20Notice%20on%20Redenomination.pdf) | 央行宣布货币改值计划，1新里昂兑换1,000旧里昂。 | The central bank announced a redenomination plan: 1 new leone would equal 1,000 old leones. |
| `D_SDR_ALLOCATION_2021` | [2021-08-23](https://www.imf.org/en/topics/special-drawing-right/2021-sdr-allocation) | IMF分配约1.988亿特别提款权（SDR），补充塞拉利昂国际储备。 | The IMF allocated about SDR 198.8 million to supplement Sierra Leone’s international reserves. |
| `D_IMF_ECF_REVIEW_A_2023` | [2023-06-05](https://www.imf.org/en/news/articles/2023/06/05/pr23196-sierra-leone-imf-exec-board-completes-6th-7th-rev-under-ecf-arrangement) | IMF完成ECF第6、7次审查，批准2,070万美元拨款。 | The IMF completed the sixth and seventh ECF reviews and approved a $20.7 million disbursement. |
| `D_IMF_ECF_REVIEW_B_2023` | [2023-11-20](https://www.imf.org/en/news/articles/2023/11/20/pr23403-sierra-leone-imf-exec-board-completes-8th-final-rev-ecf-arr) | IMF完成ECF第8次暨最终审查，批准2,070万美元拨款。 | The IMF completed the eighth and final ECF review and approved a $20.7 million disbursement. |


针对三个新的宏观数据，下面将逐一进行重复的测试与回归，包括：
- 数据可视化图表，滚动统计值（rolling mean/std）
- 因子的基本统计数据（包括均值，中位数，最大值，最小值，标准差）
- ADF与KPSS检验，检测差分之后的数据是否仍然不平稳
- 三个宏观数据，以及外汇储备，政府债务/GDP，一共五个数据之间的双因子相关性矩阵
- 五个宏观因子之间的VIF多重共线性检测
- 回归所有宏观因子与政策事件dummies，并检查hypothesis test
- 检查残差自相关，异方差，残差与汇率相关性

For the three new macro data series, the following repeated tests and regressions will be conducted one by one, including:

- Data visualization charts and rolling statistics (rolling mean/std)
- Basic statistics of the factors (including mean, median, maximum, minimum, and standard deviation)
- ADF and KPSS tests to check whether the differenced data remain non-stationary
- Pairwise correlation matrix among the three macro data series plus foreign exchange reserves and government debt/GDP, five series in total
- VIF multicollinearity test among the five macro factors
- Regress all macro factors on policy event dummies and check hypothesis tests
- Check residual autocorrelation, heteroskedasticity, and correlation between residuals and the exchange rate